# 01 – Data Acquisition
## CIP_FS26_203 | Weather & Electricity Prices in Switzerland

This notebook acquires raw data from three sources:
- **Open-Meteo API** – Historical weather data (Zurich, Basel, Geneva, Lugano)
- **ENTSO-E API** – Day-ahead electricity prices (CH bidding zone, EUR/MWh)
- **Swissgrid** – National electricity load data (.xlsx files)

In [1]:
import requests
import pandas as pd
import numpy as np
from datetime import datetime
import os

print("All packages loaded successfully!")

All packages loaded successfully!


## 1.1 Open-Meteo – Weather Data Acquisition

In [16]:
import requests
import pandas as pd
import time
import os

# --- City coordinates (representative climate regions of Switzerland) ---
cities = {
    "Zurich":  {"lat": 47.3769, "lon": 8.5417},  # Swiss Plateau
    "Basel":   {"lat": 47.5596, "lon": 7.5886},  # Jura / Northwestern Switzerland
    "Geneva":  {"lat": 46.2044, "lon": 6.1432},  # Western Switzerland
    "Lugano":  {"lat": 46.0037, "lon": 8.9511},  # South side of the Alps
}

# --- Daily weather variables to retrieve from Open-Meteo ---
variables = [
    "temperature_2m_mean",
    "temperature_2m_max",
    "temperature_2m_min",
    "precipitation_sum",
    "windspeed_10m_max",
    "shortwave_radiation_sum",
    "cloudcover_mean",
    "sunshine_duration",
]

weather_path = "../data/raw/weather/weather_raw.csv"
os.makedirs(os.path.dirname(weather_path), exist_ok=True)

# --- Skip API calls if raw file already exists ---
if os.path.exists(weather_path):
    df_weather_raw = pd.read_csv(weather_path, parse_dates=["date"])
    print(f"Weather raw data loaded from file ({len(df_weather_raw)} rows)")

else:
    # --- Fetch data for each city via Open-Meteo Historical Weather API ---
    dfs = []
    for city, coords in cities.items():
        url = "https://archive-api.open-meteo.com/v1/archive"
        params = {
            "latitude":   coords["lat"],
            "longitude":  coords["lon"],
            "start_date": "2015-01-01",
            "end_date":   "2025-12-31",
            "daily":      variables,
            "timezone":   "Europe/Zurich",
        }

        # --- Parse JSON response into DataFrame ---
        response = requests.get(url, params=params )
        response.raise_for_status()
        data = response.json()

        df = pd.DataFrame(data["daily"])
        df["date"] = pd.to_datetime(df["time"])
        df = df.drop(columns=["time"])
        df["city"] = city
        dfs.append(df)
        print(f"{city}: {len(df)} rows fetched")

        # --- Pause between requests to avoid rate limiting ---
        time.sleep(10)

    # --- Combine all cities and save to CSV ---
    df_weather_raw = pd.concat(dfs, ignore_index=True)
    df_weather_raw.to_csv(weather_path, index=False)
    print(f"\nWeather raw data saved ({len(df_weather_raw)} rows)")

df_weather_raw.head()

Weather raw data loaded from file (16072 rows)


,temperature_2m_mean,temperature_2m_max,temperature_2m_min,precipitation_sum,windspeed_10m_max,shortwave_radiation_sum,cloudcover_mean,sunshine_duration,date,city
0,4.0,7.1,0.1,0.0,6.3,6.18,7,30686.13,2015-01-01,Zurich
1,6.4,11.9,0.8,0.0,10.5,5.95,71,25552.94,2015-01-02,Zurich
2,9.4,12.7,7.5,0.0,12.3,5.50,56,28179.38,2015-01-03,Zurich
3,7.6,12.1,2.7,0.0,13.0,6.24,13,30869.75,2015-01-04,Zurich
4,6.7,10.3,2.2,0.0,12.2,6.63,6,30941.12,2015-01-05,Zurich


## 1.2 Open-Meteo – Apparent Temperature Data Acquisition

In [1]:
import requests
import pandas as pd
import time
import os

# --- Same city coordinates as weather section ---
cities = {
    "Zurich":  {"lat": 47.3769, "lon": 8.5417},
    "Basel":   {"lat": 47.5596, "lon": 7.5886},
    "Geneva":  {"lat": 46.2044, "lon": 6.1432},
    "Lugano":  {"lat": 46.0037, "lon": 8.9511},
}

# --- Define output path and create folder if it doesn't exist ---
apparent_temp_path = "../data/raw/weather/apparent_temp_raw.csv"
os.makedirs(os.path.dirname(apparent_temp_path), exist_ok=True)

# --- Skip API calls if raw file already exists (avoids unnecessary requests) ---
if os.path.exists(apparent_temp_path):
    df_apparent_raw = pd.read_csv(apparent_temp_path, parse_dates=["date"])
    print(f"Apparent temperature raw data loaded from file ({len(df_apparent_raw)} rows)")

else:
    dfs = []
    for city, coords in cities.items():

        # --- Define API endpoint and request parameters ---
        url = "https://archive-api.open-meteo.com/v1/archive"
        params = {
            "latitude":   coords["lat"],
            "longitude":  coords["lon"],
            "start_date": "2015-01-01",
            "end_date":   "2025-12-31",
            # --- Request the three apparent temperature variables ---
            "daily": [
                "apparent_temperature_mean",
                "apparent_temperature_max",
                "apparent_temperature_min"
            ],
            "timezone": "Europe/Zurich",
        }

        # --- Send GET request to API and raise error if request fails ---
        response = requests.get(url, params=params)
        response.raise_for_status()

        # --- Parse JSON response into a DataFrame ---
        data = response.json()
        df = pd.DataFrame(data["daily"])

        # --- Convert date column from string to datetime ---
        df["date"] = pd.to_datetime(df["time"])
        df = df.drop(columns=["time"])

        # --- Add city name as column to identify the source city later ---
        df["city"] = city
        dfs.append(df)
        print(f"{city}: {len(df)} rows fetched")

        # --- Pause between requests to avoid rate limiting ---
        time.sleep(10)

    # --- Combine all four cities into one DataFrame and save to CSV ---
    df_apparent_raw = pd.concat(dfs, ignore_index=True)
    df_apparent_raw.to_csv(apparent_temp_path, index=False)
    print(f"\nApparent temperature raw data saved ({len(df_apparent_raw)} rows)")

df_apparent_raw.head()

Zurich: 4018 rows fetched
Basel: 4018 rows fetched
Geneva: 4018 rows fetched
Lugano: 4018 rows fetched

Apparent temperature raw data saved (16072 rows)


,apparent_temperature_mean,apparent_temperature_max,apparent_temperature_min,date,city
0,-5.7,-0.9,-10.3,2015-01-01,Zurich
1,-3.2,1.1,-8.2,2015-01-02,Zurich
2,1.1,4.9,-1.3,2015-01-03,Zurich
3,0.4,4.5,-1.8,2015-01-04,Zurich
4,-0.8,2.3,-3.3,2015-01-05,Zurich


## 2. ENTSO-E – Day-Ahead Electricity Prices (CH)

In [17]:
import os
import pandas as pd
from entsoe import EntsoePandasClient
from dotenv import load_dotenv

# --- Load API key from .env file ---
load_dotenv()
api_key = os.getenv("ENTSOE_API_KEY")

# --- Define output path ---
prices_path = "../data/raw/prices/entsoe_prices_raw.csv"
os.makedirs(os.path.dirname(prices_path), exist_ok=True)

# --- Skip API call if raw file already exists ---
if os.path.exists(prices_path):
    df_prices_raw = pd.read_csv(prices_path, parse_dates=["date"])
    print(f"ENTSO-E prices loaded from file ({len(df_prices_raw)} rows)")

else:
    # --- Initialize ENTSO-E client ---
    client = EntsoePandasClient(api_key=api_key)

    # --- Define time range and bidding zone (Switzerland) ---
    start = pd.Timestamp("2015-01-01", tz="Europe/Zurich")
    end   = pd.Timestamp("2025-12-31", tz="Europe/Zurich")
    country_code = "CH"

    # --- Fetch day-ahead prices ---
    print("Fetching ENTSO-E day-ahead prices for CH (2015–2025)...")
    ts = client.query_day_ahead_prices(country_code, start=start, end=end)

    # --- Convert to daily average and store as DataFrame ---
    df_prices_raw = ts.resample("D").mean().reset_index()
    df_prices_raw.columns = ["date", "price_eur_mwh"]
    df_prices_raw["date"] = df_prices_raw["date"].dt.tz_localize(None)

    # --- Save to CSV ---
    df_prices_raw.to_csv(prices_path, index=False)
    print(f"ENTSO-E prices saved ({len(df_prices_raw)} rows)")

df_prices_raw.head()

ENTSO-E prices loaded from file (4017 rows)


,date,price_eur_mwh
0,2015-01-01,36.633750
1,2015-01-02,41.610000
2,2015-01-03,36.589583
3,2015-01-04,31.723333
4,2015-01-05,47.389167


## 3. Swissgrid – National Electricity Load Data

In [18]:
import requests
import os

# --- Define output directory ---
output_dir = "../data/raw/load/"
os.makedirs(output_dir, exist_ok=True)

# --- Correct URL mapping per year (2015-2019: .xls, 2020-2025: .xlsx) ---
# Base URL: https://www.swissgrid.ch/content/dam/dataimport/energy-statistic/
base_url = "https://www.swissgrid.ch/content/dam/dataimport/energy-statistic/"

year_urls = {
    2015: f"{base_url}EnergieUebersichtCH-2015.xls",
    2016: f"{base_url}EnergieUebersichtCH-2016.xls",
    2017: f"{base_url}EnergieUebersichtCH-2017.xls",
    2018: f"{base_url}EnergieUebersichtCH-2018.xls",
    2019: f"{base_url}EnergieUebersichtCH-2019.xls",
    2020: f"{base_url}EnergieUebersichtCH-2020.xlsx",
    2021: f"{base_url}EnergieUebersichtCH-2021.xlsx",
    2022: f"{base_url}EnergieUebersichtCH-2022.xlsx",
    2023: f"{base_url}EnergieUebersichtCH-2023.xlsx",
    2024: f"{base_url}EnergieUebersichtCH-2024.xlsx",
    2025: f"{base_url}EnergieUebersichtCH-2025.xlsx",
}

# --- Download each yearly file (skip if already exists) ---
for year, url in year_urls.items():
    ext = ".xls" if year <= 2019 else ".xlsx"
    filename = os.path.join(output_dir, f"swissgrid_load_{year}{ext}")

    # --- Skip download if file already exists ---
    if os.path.exists(filename):
        print(f"Already exists, skipping: swissgrid_load_{year}{ext}")
        continue

    response = requests.get(url, timeout=30)
    if response.status_code == 200:
        with open(filename, "wb") as f:
            f.write(response.content)
        print(f"Downloaded: swissgrid_load_{year}{ext} ({len(response.content) / 1024:.1f} KB)")
    else:
        print(f"Failed: {year} – HTTP {response.status_code}")

Already exists, skipping: swissgrid_load_2015.xls
Already exists, skipping: swissgrid_load_2016.xls
Already exists, skipping: swissgrid_load_2017.xls
Already exists, skipping: swissgrid_load_2018.xls
Already exists, skipping: swissgrid_load_2019.xls
Already exists, skipping: swissgrid_load_2020.xlsx
Already exists, skipping: swissgrid_load_2021.xlsx
Already exists, skipping: swissgrid_load_2022.xlsx
Already exists, skipping: swissgrid_load_2023.xlsx
Already exists, skipping: swissgrid_load_2024.xlsx
Already exists, skipping: swissgrid_load_2025.xlsx


## 4. Summary of acquired raw data

In [20]:
import os

# --- Summary of all acquired raw data ---
for folder in ["../data/raw/weather/", "../data/raw/load/", "../data/raw/prices/"]:
    files = [f for f in os.listdir(folder) if not f.startswith(".")]  # .gitkeep ausblenden
    print(f"{folder}: {len(files)} file(s)")
    for f in sorted(files):
        print(f"  - {f}")

../data/raw/weather/: 1 file(s)
  - weather_raw.csv
../data/raw/load/: 11 file(s)
  - swissgrid_load_2015.xls
  - swissgrid_load_2016.xls
  - swissgrid_load_2017.xls
  - swissgrid_load_2018.xls
  - swissgrid_load_2019.xls
  - swissgrid_load_2020.xlsx
  - swissgrid_load_2021.xlsx
  - swissgrid_load_2022.xlsx
  - swissgrid_load_2023.xlsx
  - swissgrid_load_2024.xlsx
  - swissgrid_load_2025.xlsx
../data/raw/prices/: 1 file(s)
  - entsoe_prices_raw.csv
